# Simplified Copy-Trade Backtest

Simulates copying trades of a hand-picked set of wallets against the
processed Polygon trade shards.

## Strategy
- When a watched wallet prints a **BUY** on token `(condition_id, outcome)` at price `p`
  we place a buy order at a **slightly worse** price: `order_price = clip(p + WORSE_PRICE_OFFSET, 0.001, 0.999)`.
- The order is matched against the **first subsequent trade** whose effective price is
  `<= order_price` within `FILL_HORIZON_SECONDS`.
  - *same-token* liquidity: later BUY trades on the same `(condition_id, outcome)`.
  - *opposite-token* liquidity: later SELL trades on the **complementary** outcome
    (binary markets only), with effective price = `1 - sell_price`.
- For a wallet **SELL** the mirror applies: order at `p - WORSE_PRICE_OFFSET`, filled
  by the first later trade with effective price `>= order_price`.
  - same-token: later SELL prints on the same outcome.
  - opposite-token: later BUY prints on the complementary outcome, effective price `1 - buy_price`.

## Sharding
Trades are partitioned by the first hex character of `condition_id` (after `0x`).
All shards are processed in parallel; within each shard the backtest is exact
because a `condition_id` always falls in exactly one shard file.

## Outputs
One event ledger DataFrame with columns:
- `row_type` — `trigger` (watched-wallet trade that generated an order) or `fill` (our fill).
- `order_id` — unique integer identifying the copy order.
- `wallet` — originating wallet (`fill` rows carry the trigger wallet for reference).
- `condition_id`, `outcome`, `side`, `dt`, `tx_hash`, `price`.
- `order_price` — the price our order was placed at (`NaN` for fill rows).
- `fill_source` — `same_token` / `opposite_token` / `NaN` for trigger rows.
- `token_winner` — market resolution flag.
- `fill_pnl_usdc` — PnL of *our* position on fill rows (NaN for trigger rows),
  computed as stake × ( 1/exec_price − 1 ) for winning tokens, −stake for losers.

## Visualisation
Cumulative PnL chart with two series:
- **Wallet cohort** — raw `trade_pnl` of the watched wallets.
- **Copy strategy** — `fill_pnl_usdc` of our simulated fills.


## Configuration

In [171]:
%load_ext autoreload
%autoreload 2

import datetime
from pathlib import Path

# ── Input wallets ─────────────────────────────────────────────────────────────
# Replace with any wallet addresses you want to copy.
# Example: load from a CSV, a workspace registry, or define inline.
WATCHED_WALLETS: list[str] = None

ONLY_BUY_TRIGGERS = True
MAX_EXPOSURE_PER_WALLET_1h = 100

# ── Test period ───────────────────────────────────────────────────────────────
# None = use entire dataset (start/end are inferred from the data).
# Set to datetime.date objects to restrict the window.
PERIOD_START: datetime.date | None = datetime.date(2026, 2, 16)
# PERIOD_START: datetime.date | None = datetime.date(2026, 1, 1) # None   # e.g. datetime.date(2026, 2, 16)
PERIOD_END:   datetime.date | None = datetime.date(2026, 3, 20)
# PERIOD_END:   datetime.date | None = datetime.date(2026, 2, 16) # None   # e.g. datetime.date(2026, 3, 11)

# ── Copy-trade execution parameters ──────────────────────────────────────────
WORSE_PRICE_OFFSET: float = 0
FILL_HORIZON_SECONDS: float = 300     # max seconds after trigger to search for a fill
STAKE_USDC: float = 100.0               # max USDC per order (order qty = min(trigger_qty, STAKE_USDC / order_price))
FEE_BPS: float = 10.0                   # taker fee in basis points

# ── Data ─────────────────────────────────────────────────────────────────────
TRADES_DIR     = Path("../../data/polygon_trades_processed")
RAW_TRADES_DIR = Path("../../data/trades_polygon")

# ── Parallelism ───────────────────────────────────────────────────────────────
MAX_WORKERS: int = 10   # number of threads for parallel shard processing

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [172]:
banned_wallets = set()
if(WATCHED_WALLETS is not None):
    WATCHED_WALLETS = [w for w in WATCHED_WALLETS if w not in banned_wallets]

## Optionally load wallets from stage-1 workspace

If `WATCHED_WALLETS` is empty above, this cell loads the wallet set from the
stage-1 strategy registry. Skip or modify as needed.

In [173]:
import pandas as pd

if not WATCHED_WALLETS:
    WORKSPACE_DIR = Path("../../data/trade_signals_workspace_v2")
    wallets_path = WORKSPACE_DIR / "selected_wallets_v2.parquet"
    if wallets_path.exists():
        _wallets_df = pd.read_parquet(wallets_path, columns=["wallet"])
        WATCHED_WALLETS = _wallets_df["wallet"].tolist()
        print(f"Loaded {len(WATCHED_WALLETS)} wallets from {wallets_path.name}")
    else:
        print("No wallet source found — set WATCHED_WALLETS manually or run stage1 first.")
else:
    print(f"Using {len(WATCHED_WALLETS)} manually specified wallets.")

Loaded 41 wallets from selected_wallets_v2.parquet


## Discover shards and derive test period

In [174]:
import pandas as pd

SHARD_PATHS: list[Path] = sorted(TRADES_DIR.glob("*.parquet"))
print(f"Found {len(SHARD_PATHS)} shards under {TRADES_DIR}")

# Derive END_DATE_TRAIN from the is_train flag (last date where is_train=True).
# Test data is everything strictly after END_DATE_TRAIN.
_sample = pd.read_parquet(SHARD_PATHS[0], columns=["dt", "is_train"])
_sample["dt"] = pd.to_datetime(_sample["dt"], utc=True)
END_DATE_TRAIN: datetime.date = _sample.loc[_sample["is_train"], "dt"].max().date()
_data_end: datetime.date      = _sample["dt"].max().date()
del _sample
print(f"END_DATE_TRAIN (last train date) : {END_DATE_TRAIN}")

# Backtest always runs on test data only (strictly after END_DATE_TRAIN).
# PERIOD_START / PERIOD_END can narrow the window further, but cannot go
# earlier than the day after END_DATE_TRAIN.
#_test_start = END_DATE_TRAIN + datetime.timedelta(days=1)
period_start: datetime.date = PERIOD_START # datetime.date(2026, 2, 16) #  max(PERIOD_START, _test_start) if PERIOD_START is not None else _test_start
period_end:   datetime.date = PERIOD_END #if PERIOD_END is not None else _data_end
print(f"Backtest period : {period_start}  →  {period_end}")

Found 16 shards under ../../data/polygon_trades_processed
END_DATE_TRAIN (last train date) : 2026-02-15
Backtest period : 2026-02-16  →  2026-03-20


## Backtest engine (per-shard)

Each shard is processed independently:
1. Load only rows within the backtest period (test data only).
2. Identify trigger events (watched-wallet trades).
3. Build a per-`condition_id` opposite-outcome map (binary markets only).
4. Process triggers in time order, maintaining a copy-portfolio **position** per `(wallet, condition_id, outcome)`.
5. For each trigger, place a copy order:
   - **BUY**: `qty_ordered = min(trig_qty, STAKE_USDC / order_price)` — worst-case loss = `qty × order_price ≤ STAKE_USDC`
   - **SELL**: `qty_ordered = min(trig_qty, position, STAKE_USDC / (1 − order_price))` — worst-case loss = `qty × (1 − order_price) ≤ STAKE_USDC`. Skipped if position = 0 (no short selling).
6. Scan tape prints in time order within `FILL_HORIZON_SECONDS`. Each matching print partially fills the order: `fill_qty = min(remaining_qty, tape_qty)`.
7. One **fill** ledger row is emitted per partial fill. BUY fills increment position; SELL fills decrement it.

`fill_pnl_usdc` per fill row (all executed at `exec_price = order_price`, limit fill):
- **BUY winner**: `fill_qty × (1 − exec_price) − fill_qty × exec_price × fee`
- **BUY loser**:  `−fill_qty × exec_price × (1 + fee)`
- **SELL winner** (sold below par): `fill_qty × (exec_price − 1) − fill_qty × exec_price × fee`
- **SELL loser** (sold above par):  `fill_qty × exec_price × (1 − fee)`


In [175]:
import numpy as np
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

# ─────────────────────────────────────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────────────────────────────────────

def _build_complement_map(df: pd.DataFrame) -> dict[tuple[str, str], str]:
    """Return {(condition_id, outcome): opposite_outcome} for binary markets."""
    pairs: dict[str, set] = {}
    for cid, grp in df.groupby("condition_id", sort=False):
        pairs[cid] = set(grp["outcome"].dropna().unique())
    result: dict[tuple[str, str], str] = {}
    for cid, outcomes in pairs.items():
        if len(outcomes) == 2:
            a, b = sorted(outcomes)
            result[(cid, a)] = b
            result[(cid, b)] = a
    return result


def _simulate_shard(
    shard_path: Path,
    raw_tape_path: Path,
    watched_wallets: set[str],
    period_start: datetime.date,
    period_end: datetime.date,
    worse_price_offset: float,
    fill_horizon_seconds: float,
    stake_usdc: float,
    fee_bps: float,
) -> pd.DataFrame:
    """Process one shard file and return an event ledger DataFrame.

    Triggers are read from ``shard_path`` (processed per-wallet aggregated trades),
    which supplies ``token_winner`` and ``trade_pnl``.

    The fill tape is read from ``raw_tape_path`` (raw on-chain individual fills).
    Orders are simulated chronologically against the tape so that each tape print's
    quantity can only be consumed once across all active copy orders in the shard.
    """
    TRIG_COLS = [
        "wallet", "condition_id", "outcome", "dt", "side",
        "avg_price", "total_quantity", "trade_pnl", "token_winner",
        "tx_hash",
    ]
    tdf = pd.read_parquet(shard_path, columns=TRIG_COLS)
    if tdf.empty:
        return pd.DataFrame()

    tdf["dt"] = pd.to_datetime(tdf["dt"], utc=True)
    tdf = tdf.rename(columns={"avg_price": "price", "total_quantity": "quantity"})

    date_mask = (
        (tdf["dt"].dt.date >= period_start)
        & (tdf["dt"].dt.date <= period_end)
    )
    tdf = tdf[date_mask].copy()
    if tdf.empty:
        return pd.DataFrame()

    tdf["price"] = tdf["price"].astype(float)
    tdf["quantity"] = tdf["quantity"].astype(float)

    if ONLY_BUY_TRIGGERS:
        trigger_mask = (tdf["wallet"].isin(watched_wallets)) & (tdf["side"] == "BUY")
    else:
        trigger_mask = tdf["wallet"].isin(watched_wallets)
    triggers_df = tdf[trigger_mask].copy()
    if triggers_df.empty:
        return pd.DataFrame()

    tw_map: dict[tuple[str, str], bool] = {}
    for row in tdf[["condition_id", "outcome", "token_winner"]].itertuples(index=False):
        key = (row.condition_id, row.outcome)
        if key not in tw_map and row.token_winner is not None:
            tw_map[key] = bool(row.token_winner)

    complement = _build_complement_map(tdf)
    triggers_df = triggers_df.sort_values("dt", kind="mergesort").reset_index(drop=True)

    TAPE_COLS = ["condition_id", "outcome", "block_timestamp", "side", "price", "quantity", "tx_hash", "token_winner"]
    if raw_tape_path.exists():
        rdf = pd.read_parquet(raw_tape_path, columns=TAPE_COLS)
    else:
        rdf = pd.DataFrame(columns=TAPE_COLS)

    if rdf.empty:
        ledger_rows = []
        for trig in triggers_df.itertuples(index=False):
            cid = trig.condition_id
            outcome = trig.outcome
            side = trig.side
            trig_dt = trig.dt
            trig_px = float(trig.price)
            trig_qty = float(trig.quantity)
            trig_tw = bool(trig.token_winner)
            wallet = trig.wallet
            if side == "BUY":
                order_px = float(np.clip(trig_px + worse_price_offset, 0.001, 0.999))
                qty_ordered = min(trig_qty, stake_usdc / order_px)
            else:
                order_px = float(np.clip(trig_px - worse_price_offset, 0.001, 0.999))
                qty_ordered = trig_qty
            ledger_rows.append({
                "row_type": "trigger", "order_id": len(ledger_rows) + 1,
                "wallet": wallet, "condition_id": cid, "outcome": outcome,
                "side": side, "dt": trig_dt, "tx_hash": trig.tx_hash,
                "price": trig_px, "order_price": order_px,
                "qty_ordered": qty_ordered, "qty_filled": 0.0,
                "fill_qty": None, "fill_source": None,
                "token_winner": trig_tw, "exec_price": None,
                "fill_pnl_usdc": None,
                "wallet_trade_pnl": float(trig.trade_pnl) if trig.trade_pnl is not None else None,
            })
        result = pd.DataFrame(ledger_rows)
        result["shard"] = shard_path.stem
        return result

    rdf["dt"] = pd.to_datetime(rdf["block_timestamp"], unit="s", utc=True)
    rdf = rdf.drop(columns=["block_timestamp"])

    tape_start = pd.Timestamp(period_start, tz="UTC")
    tape_end = pd.Timestamp(period_end, tz="UTC") + pd.Timedelta(days=1)
    rdf = rdf[(rdf["dt"] >= tape_start) & (rdf["dt"] < tape_end)].copy()
    rdf["price"] = rdf["price"].astype(float)
    rdf["quantity"] = rdf["quantity"].astype(float)
    rdf = rdf.sort_values("dt", kind="mergesort").reset_index(drop=True)

    fee = fee_bps / 10_000.0
    horizon = pd.Timedelta(seconds=fill_horizon_seconds)
    eps = 1e-12

    ledger_rows: list[dict] = []
    order_counter = 0
    positions: dict[tuple[str, str, str], float] = defaultdict(float)

    orders: dict[int, dict] = {}
    books: dict[tuple[str, str, str], list[dict]] = defaultdict(list)

    def _append_trigger_row(order_id: int, trig, order_px: float, qty_ordered: float, trig_tw: bool) -> None:
        ledger_rows.append({
            "row_type": "trigger",
            "order_id": order_id,
            "wallet": trig.wallet,
            "condition_id": trig.condition_id,
            "outcome": trig.outcome,
            "side": trig.side,
            "dt": trig.dt,
            "tx_hash": trig.tx_hash,
            "price": float(trig.price),
            "order_price": order_px,
            "qty_ordered": qty_ordered,
            "qty_filled": None,
            "fill_qty": None,
            "fill_source": None,
            "token_winner": trig_tw,
            "exec_price": None,
            "fill_pnl_usdc": None,
            "wallet_trade_pnl": float(trig.trade_pnl) if trig.trade_pnl is not None else None,
        })

    # order can match with trade on the same side and token, or opposite side and opposite token    
    def _register_order(order_id: int, order: dict, trig_tw: bool) -> None:
        cid = order["condition_id"]
        outcome = order["outcome"]
        side = order["side"]
        books[(cid, outcome, side)].append({
            "order_id": order_id,
            "fill_source": "same_token",
            "fill_tw": trig_tw,
            "opposite": False,
        })
        opp_outcome = complement.get((cid, outcome))
        if opp_outcome is None:
            return
        opp_tw = tw_map.get((cid, opp_outcome), not trig_tw)
        opp_tape_side = "SELL" if side == "BUY" else "BUY"
        books[(cid, opp_outcome, opp_tape_side)].append({
            "order_id": order_id,
            "fill_source": "opposite_token",
            "fill_tw": trig_tw,
            "opposite": True,
        })

    def _process_tape_row(row) -> None:
        key = (row.condition_id, row.outcome, row.side)
        entries = books.get(key)
        if not entries:
            return

        tape_dt = row.dt
        tape_price = float(row.price)
        tape_remaining = float(row.quantity)
        if tape_remaining <= eps:
            return

        survivors: list[dict] = []
        for entry in entries:
            order = orders.get(entry["order_id"])
            if order is None:
                continue
            if order["remaining_qty"] <= eps:
                continue
            if order["deadline"] < tape_dt:
                continue

            eff_price = float(np.clip(1.0 - tape_price, 0.001, 0.999)) if entry["opposite"] else tape_price
            eligible = eff_price <= order["order_price"] if order["side"] == "BUY" else eff_price >= order["order_price"]

            if eligible and tape_remaining > eps:
                fill_qty = min(order["remaining_qty"], tape_remaining)
                tape_remaining -= fill_qty
                order["remaining_qty"] -= fill_qty

                if order["side"] == "BUY":
                    gross = fill_qty * (1.0 - order["order_price"]) if entry["fill_tw"] else -fill_qty * order["order_price"]
                    fill_pnl = gross - fill_qty * order["order_price"] * fee
                    positions[order["pos_key"]] += fill_qty
                else:
                    gross = fill_qty * (order["order_price"] - 1.0) if entry["fill_tw"] else fill_qty * order["order_price"]
                    fill_pnl = gross - fill_qty * order["order_price"] * fee
                    positions[order["pos_key"]] = max(positions[order["pos_key"]] - fill_qty, 0.0)

                ledger_rows.append({
                    "row_type": "fill",
                    "order_id": entry["order_id"],
                    "wallet": order["wallet"],
                    "condition_id": order["condition_id"],
                    "outcome": order["outcome"],
                    "side": order["side"],
                    "dt": tape_dt,
                    "tx_hash": row.tx_hash,
                    "price": eff_price,
                    "order_price": None,
                    "qty_ordered": order["qty_ordered"],
                    "qty_filled": None,
                    "fill_qty": fill_qty,
                    "fill_source": entry["fill_source"],
                    "token_winner": entry["fill_tw"],
                    "exec_price": order["order_price"],
                    "fill_pnl_usdc": fill_pnl,
                    "wallet_trade_pnl": None,
                })

            if order["remaining_qty"] > eps and order["deadline"] >= tape_dt:
                survivors.append(entry)

        if survivors:
            books[key] = survivors
        else:
            books.pop(key, None)

    tape_iter = iter(rdf[["condition_id", "outcome", "dt", "side", "price", "quantity", "tx_hash", "token_winner"]].itertuples(index=False))
    current_tape = next(tape_iter, None)

    for trig in triggers_df.itertuples(index=False):
        while current_tape is not None and current_tape.dt <= trig.dt:
            _process_tape_row(current_tape)
            current_tape = next(tape_iter, None)

        cid = trig.condition_id
        outcome = trig.outcome
        side = trig.side
        trig_px = float(trig.price)
        trig_qty = float(trig.quantity)
        trig_tw = bool(trig.token_winner)
        wallet = trig.wallet
        pos_key = (wallet, cid, outcome)

        if side == "BUY":
            order_px = float(np.clip(trig_px + worse_price_offset, 0.001, 0.999))
            qty_ordered = min(trig_qty, stake_usdc / order_px)
        else:
            order_px = float(np.clip(trig_px - worse_price_offset, 0.001, 0.999))
            current_pos = positions.get(pos_key, 0.0)
            if current_pos <= eps:
                continue
            sell_cap = stake_usdc / max(1.0 - order_px, 1e-9)
            qty_ordered = min(trig_qty, current_pos, sell_cap)
            if qty_ordered <= eps:
                continue

        order_counter += 1
        order_id = order_counter

        order = {
            "wallet": wallet,
            "condition_id": cid,
            "outcome": outcome,
            "side": side,
            "order_price": order_px,
            "qty_ordered": qty_ordered,
            "remaining_qty": qty_ordered,
            "deadline": trig.dt + horizon,
            "pos_key": pos_key,
        }
        orders[order_id] = order

        _append_trigger_row(order_id, trig, order_px, qty_ordered, trig_tw)
        _register_order(order_id, order, trig_tw)

    while current_tape is not None:
        _process_tape_row(current_tape)
        current_tape = next(tape_iter, None)

    if not ledger_rows:
        return pd.DataFrame()

    result = pd.DataFrame(ledger_rows)
    result["shard"] = shard_path.stem

    filled_by_order = (
        result[result["row_type"] == "fill"]
        .groupby("order_id")["fill_qty"]
        .sum()
        .rename("_total_filled")
    )
    result = result.merge(filled_by_order, on="order_id", how="left")
    trig_mask = result["row_type"] == "trigger"
    result.loc[trig_mask, "qty_filled"] = result.loc[trig_mask, "_total_filled"].fillna(0.0)
    result = result.drop(columns=["_total_filled"])
    result["qty_filled"] = result["qty_filled"].astype(float)

    return result



## Run backtest across all shards (parallel)

In [176]:
assert WATCHED_WALLETS, "WATCHED_WALLETS is empty — set wallets in the config cell or run the wallet-load cell."

watched_set = set(WATCHED_WALLETS)
shard_results: list[pd.DataFrame] = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(
            _simulate_shard,
            path,
            RAW_TRADES_DIR / path.name,
            watched_set,
            period_start,
            period_end,
            WORSE_PRICE_OFFSET,
            FILL_HORIZON_SECONDS,
            STAKE_USDC,
            FEE_BPS,
        ): path
        for path in SHARD_PATHS
    }
    for future in as_completed(futures):
        path = futures[future]
        try:
            df = future.result()
            if df is not None and not df.empty:
                shard_results.append(df)
        except Exception as exc:
            print(f"Shard {path.name} raised an exception: {exc}")
            raise

if shard_results:
    event_ledger: pd.DataFrame = (
        pd.concat(shard_results, ignore_index=True)
        .sort_values(["dt", "shard", "order_id", "row_type"])
        .reset_index(drop=True)
    )
    _key = event_ledger[["shard", "order_id"]]
    _pairs = _key.drop_duplicates().reset_index(drop=True)
    _pairs["global_order_id"] = _pairs.index + 1
    event_ledger = event_ledger.merge(_pairs, on=["shard", "order_id"], how="left")
    event_ledger["order_id"] = event_ledger["global_order_id"]
    event_ledger = event_ledger.drop(columns=["global_order_id"])
else:
    event_ledger = pd.DataFrame(columns=[
        "row_type", "order_id", "wallet", "condition_id", "outcome",
        "side", "dt", "tx_hash", "price", "order_price",
        "qty_ordered", "qty_filled", "fill_qty",
        "fill_source", "token_winner", "exec_price", "fill_pnl_usdc",
        "wallet_trade_pnl", "shard",
    ])

triggers = event_ledger[event_ledger["row_type"] == "trigger"]
fills = event_ledger[event_ledger["row_type"] == "fill"]
filled_orders = fills["order_id"].nunique()

print(f"Trigger events    : {len(triggers):>7,}")
print(f"Fill rows         : {len(fills):>7,}")
print(f"Orders with fills : {filled_orders:>7,}")
if len(triggers) > 0:
    print(f"Order fill rate   : {filled_orders/len(triggers)*100:.1f}%")
else:
    print("No trigger events found — check WATCHED_WALLETS and the period.")



Trigger events    :  13,578
Fill rows         :  15,918
Orders with fills :   7,084
Order fill rate   : 52.2%


## Summary statistics

In [177]:
fee = FEE_BPS / 10_000.0

if not fills.empty:
    filled_copy_pnl    = fills["fill_pnl_usdc"].sum()
    total_qty_filled   = fills["fill_qty"].sum()
    total_notional     = (fills["fill_qty"] * fills["exec_price"]).sum()
    orders_with_fills  = fills["order_id"].nunique()
    partial_orders     = (fills.groupby("order_id").size() > 1).sum()
    win_rate           = fills.groupby("order_id")["token_winner"].first().mean()
    avg_exec_price     = fills["exec_price"].mean()
    fill_src_counts    = fills["fill_source"].value_counts()

    # Partial-fill statistics
    trig_qty = triggers.set_index("order_id")["qty_ordered"]
    fill_qty = triggers.set_index("order_id")["qty_filled"]
    fill_pct = (fill_qty / trig_qty.clip(lower=1e-12) * 100).replace([float('inf'), float('nan')], 0)

    print(f"=== Copy-strategy performance ===")
    print(f"  Orders triggered    : {len(triggers):>7,}")
    print(f"  Orders with fills   : {orders_with_fills:>7,}  ({orders_with_fills/len(triggers)*100:.1f}%)")
    print(f"  Orders multi-fill   : {partial_orders:>7,}  ({partial_orders/max(orders_with_fills,1)*100:.1f}% of filled)")
    print(f"  Avg fill %          : {fill_pct[fill_pct>0].mean():>7.1f}%")
    print(f"  Total qty filled    : {total_qty_filled:>10,.2f} tokens")
    print(f"  Total notional      : {total_notional:>10,.2f} USDC")
    print(f"  Net PnL (USDC)      : {filled_copy_pnl:>10,.2f}")
    print(f"  Net ROI on notional : {filled_copy_pnl/total_notional*100:>8.2f}%")
    print(f"  Win rate (by order) : {win_rate*100:>8.2f}%")
    print(f"  Avg exec price      : {avg_exec_price:>8.4f}")
    print(f"\n  Fill source breakdown:")
    for src, cnt in fill_src_counts.items():
        print(f"    {src:<20s}: {cnt:>6,}  ({cnt/len(fills)*100:.1f}%)")
else:
    print("No fills — nothing to summarise.")

# Watched-wallet cohort summary
wallet_pnl = triggers["wallet_trade_pnl"].sum()
print(f"\n=== Watched-wallet cohort ===")
print(f"  Total trades        : {len(triggers):>7,}")
print(f"  Total trade PnL     : {wallet_pnl:>10,.2f} USDC")


=== Copy-strategy performance ===
  Orders triggered    :  13,578
  Orders with fills   :   7,084  (52.2%)
  Orders multi-fill   :   3,278  (46.3% of filled)
  Avg fill %          :    88.1%
  Total qty filled    : 539,479.88 tokens
  Total notional      : 169,775.95 USDC
  Net PnL (USDC)      :  27,225.20
  Net ROI on notional :    16.04%
  Win rate (by order) :    52.95%
  Avg exec price      :   0.4615

  Fill source breakdown:
    same_token          : 12,602  (79.2%)
    opposite_token      :  3,316  (20.8%)

=== Watched-wallet cohort ===
  Total trades        :  13,578
  Total trade PnL     : 124,976.22 USDC


## Event ledger preview

In [178]:
display_cols = [
    "row_type", "order_id", "wallet", "condition_id", "outcome", "side",
    "dt", "tx_hash", "price", "order_price", "exec_price",
    "qty_ordered", "qty_filled", "fill_qty",
    "fill_source", "token_winner", "fill_pnl_usdc", "wallet_trade_pnl",
]
available = [c for c in display_cols if c in event_ledger.columns]

# Show a few trigger+fill pairs
sample_orders = event_ledger["order_id"].unique()[:5]
event_ledger[
    event_ledger["order_id"].isin(sample_orders)
][available].sort_values(["order_id", "row_type"])


,row_type,order_id,wallet,condition_id,outcome,side,dt,tx_hash,price,order_price,exec_price,qty_ordered,qty_filled,fill_qty,fill_source,token_winner,fill_pnl_usdc,wallet_trade_pnl
0,trigger,1,0x9e8ecc4cb3c4e48f544cba2fbbb252a6a65e8db8,0x23aff5caff52e134b96e2fa6a2cb8fe4f17c1a7b36f2705ba0fb352557aca75c,No,BUY,2026-02-16 00:12:03+00:00,0x19f834d33ffb6929196782523499f7a7cd5dc7566c32af85abf911514624f416,0.773000,0.773000,NaN,22.611981,0.0,NaN,None,True,NaN,5.132919
1,trigger,2,0x66c1a6fe836ff555ca32848646acedbbe93bfa3f,0x690943971c6055658aaa54649cb491c70f404c64207c413eb7720e3e825e4c45,Yes,BUY,2026-02-16 00:30:25+00:00,0x3a198da79cc87db83fd0aea97819076461bea885bd43cba07f41ef1bee3ccd78,0.400000,0.400000,NaN,200.000000,0.0,NaN,None,False,NaN,-80.000000
2,trigger,3,0x20a89c60befe1e2d866042deed93ab72be6fd960,0x5bcc2e07d8e4a2f5eff7f5292f2f6b1fe0d009da3d9a9ad47f184425f6efa6d2,Tucson Roadrunners,BUY,2026-02-16 00:42:13+00:00,0xe7f5673450f30c5beb2e8f6fb6b5d9b560b727f352181c0231dd10b570dc3b79,0.820000,0.820000,NaN,6.000000,0.0,NaN,None,True,NaN,1.080000
3,trigger,4,0x9e8ecc4cb3c4e48f544cba2fbbb252a6a65e8db8,0x23aff5caff52e134b96e2fa6a2cb8fe4f17c1a7b36f2705ba0fb352557aca75c,No,BUY,2026-02-16 00:45:31+00:00,0x4da5d4359e04c1f593766c86bf037fe70efb48c7a3d5afe878faca1eaa68620b,0.773011,0.773011,NaN,0.066078,0.0,NaN,None,True,NaN,0.014999
4,trigger,5,0x66c1a6fe836ff555ca32848646acedbbe93bfa3f,0x7147515e4478e76babb5a383eff44692059572b7b8d05d2db834e14a038e493e,No,BUY,2026-02-16 01:06:47+00:00,0xdd5bd28c86ce6e548e5cf60b5e8c247710112a6847ea13a62975fb2f7191499b,0.860000,0.860000,NaN,4.440000,0.0,NaN,None,True,NaN,0.621600


In [179]:
event_ledger[event_ledger['side'] == 'BUY'].groupby('wallet').agg(
    fill_pnl_usdc_sum=('fill_pnl_usdc', 'sum'),
    wallet_trade_pnl_sum=('wallet_trade_pnl', 'sum'),
    buy_count=('side', 'count')
)

,fill_pnl_usdc_sum,wallet_trade_pnl_sum,buy_count
wallet,,,
0x002dcd37b0b8fa8db98236e599fe1b90d6272561,-78.292158,571.112080,174
0x00ffb912f1bf01c0123869ddb92d4b6a6ad3f7ad,-5017.465271,-30872.076578,1453
0x0b652d3e55be0e1d50c2f7be39358585c415293e,30.423270,857.100485,1141
0x0c0e270cf879583d6a0142fc817e05b768d0434e,7.672593,921.307460,323
0x0cb10c40b0776e9ee8cef970af85724654dda76c,807.137790,8803.265493,950
0x1c144e30f405a25f991cbd8baa15d40599090869,-677.329401,-8946.034706,1032
0x1c1e841584db14084e10e7dca2ad0ab7b60dbfe7,722.055227,-631.627074,200
0x20a89c60befe1e2d866042deed93ab72be6fd960,347.489585,1230.116614,3219
0x210e1d92cfacc18699a2527e705c8bc6828c5fd3,-77.565304,-79.704000,78


In [180]:
event_ledger[event_ledger['side'] == 'SELL'].groupby('wallet').agg(
    fill_pnl_usdc_sum=('fill_pnl_usdc', 'sum'),
    wallet_trade_pnl_sum=('wallet_trade_pnl', 'sum'),
    sell_count=('side', 'count')
)

,fill_pnl_usdc_sum,wallet_trade_pnl_sum,sell_count
wallet,,,


In [181]:
sample_filled_orders = fills["order_id"].unique()[:50]
event_ledger[
    (event_ledger['side'] == 'SELL') &
    (event_ledger["order_id"].isin(sample_filled_orders))
].sort_values(["order_id", "dt"])

,row_type,order_id,wallet,condition_id,outcome,side,dt,tx_hash,price,order_price,qty_ordered,qty_filled,fill_qty,fill_source,token_winner,exec_price,fill_pnl_usdc,wallet_trade_pnl,shard


## Build PnL time series

In [182]:
def resample_hourly_series(df: pd.DataFrame, dt_col: str, pnl_col: str) -> pd.DataFrame:
    """Resample a PnL column to 1-h buckets and compute cumulative sum."""
    if df.empty or pnl_col not in df.columns:
        return pd.DataFrame(columns=["trade_dt", "net_pnl_usdc", "cum_net_pnl_usdc"])
    hourly = (
        df.assign(trade_dt=df[dt_col].dt.floor("1h"))
        .groupby("trade_dt", as_index=False)[pnl_col]
        .sum()
        .rename(columns={pnl_col: "net_pnl_usdc"})
        .sort_values("trade_dt")
        .reset_index(drop=True)
    )
    hourly["cum_net_pnl_usdc"] = hourly["net_pnl_usdc"].cumsum()
    return hourly


def with_zero_anchor(hourly: pd.DataFrame) -> pd.DataFrame:
    """Prepend a zero anchor one hour before the first bucket."""
    if hourly.empty:
        return hourly
    anchor = pd.DataFrame({
        "trade_dt": [hourly["trade_dt"].min() - pd.Timedelta(hours=1)],
        "net_pnl_usdc": [0.0],
        "cum_net_pnl_usdc": [0.0],
    })
    return pd.concat([anchor, hourly], ignore_index=True)


# Wallet cohort PnL: from trigger rows (their actual trade_pnl)
wallet_hourly = resample_hourly_series(triggers, "dt", "wallet_trade_pnl")

# Copy-strategy PnL: from fill rows
copy_hourly = resample_hourly_series(fills, "dt", "fill_pnl_usdc")

print(f"Wallet cohort hourly buckets : {len(wallet_hourly)}")
print(f"Copy strategy hourly buckets : {len(copy_hourly)}")

Wallet cohort hourly buckets : 773
Copy strategy hourly buckets : 705


## Cumulative PnL chart

In [183]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = go.Figure()

# ── Wallet cohort trace ───────────────────────────────────────────────────────
if not wallet_hourly.empty:
    wh = with_zero_anchor(wallet_hourly)
    fig.add_trace(go.Scatter(
        x=wh["trade_dt"],
        y=wh["cum_net_pnl_usdc"],
        mode="lines",
        line=dict(dash="dot", width=2),
        opacity=0.7,
        name="Watched wallets (raw PnL)",
        hovertemplate=(
            "<b>Watched wallets</b><br>"
            "%{x|%Y-%m-%d %H:%M}<br>"
            "cum PnL: %{y:.2f} USDC<extra></extra>"
        ),
    ))

# ── Copy-strategy trace ───────────────────────────────────────────────────────
if not copy_hourly.empty:
    ch = with_zero_anchor(copy_hourly)
    fig.add_trace(go.Scatter(
        x=ch["trade_dt"],
        y=ch["cum_net_pnl_usdc"],
        mode="lines",
        line=dict(dash="solid", width=2),
        name="Copy strategy (filled)",
        hovertemplate=(
            "<b>Copy strategy</b><br>"
            "%{x|%Y-%m-%d %H:%M}<br>"
            "cum PnL: %{y:.2f} USDC<extra></extra>"
        ),
    ))

fig.add_hline(y=0, line_dash="dash", line_color="grey", line_width=1, opacity=0.5)

fig.update_layout(
    template="plotly_white",
    height=480,
    title=dict(
        text=(
            f"Copy-trade backtest — cumulative PnL (1h) | "
            f"{period_start} → {period_end} | "
            f"{len(WATCHED_WALLETS)} wallets | "
            f"worse_price={WORSE_PRICE_OFFSET:.2f} | "
            f"horizon={int(FILL_HORIZON_SECONDS)}s"
        ),
        font=dict(size=13),
    ),
    xaxis_title="Time (1h buckets)",
    yaxis_title="Cumulative net PnL (USDC)",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)

fig.show()

## Diagnostics

In [184]:
# Fill rate by side (order-level)
if not triggers.empty:
    trig_by_side = triggers.groupby("side").size().rename("triggers")
    fill_by_side = fills.groupby("side")["order_id"].nunique().rename("orders_filled")
    fill_rows_by_side = fills.groupby("side").size().rename("fill_rows")
    side_summary = pd.concat([trig_by_side, fill_by_side, fill_rows_by_side], axis=1).fillna(0).astype(int)
    side_summary["fill_rate"] = (side_summary["orders_filled"] / side_summary["triggers"] * 100).round(1)
    display(side_summary)


,triggers,orders_filled,fill_rows,fill_rate
side,,,,
BUY,13578,7084,15918,52.2


In [185]:
# Fill source breakdown by side
if not fills.empty:
    display(
        fills.groupby(["side", "fill_source"]).size()
        .rename("count")
        .reset_index()
        .sort_values(["side", "fill_source"])
    )

,side,fill_source,count
0,BUY,opposite_token,3316
1,BUY,same_token,12602


In [186]:
df = event_ledger.assign(
    is_trigger=event_ledger["row_type"].eq("trigger"),
    is_fill=event_ledger["row_type"].eq("fill"),
)

# --- wallet-level stats (row-based) ---
wallet_stats = (
    df.groupby("wallet")
    .agg(
        total_triggers=("is_trigger", "sum"),
        total_fills=("is_fill", "sum"),
        total_fill_pnl=("fill_pnl_usdc", "sum"),
        total_trade_pnl=("wallet_trade_pnl", "sum"),
        total_pnl=("fill_pnl_usdc", "sum"),
    )
)

# --- order-level logic (trigger -> fill) ---
order_stats = (
    df.groupby(["wallet", "order_id"])
    .agg(
        has_trigger=("is_trigger", "any"),
        has_fill=("is_fill", "any"),
    )
    .assign(trigger_with_fill=lambda x: x["has_trigger"] & x["has_fill"])
    .groupby("wallet")
    .agg(triggers_with_fill=("trigger_with_fill", "sum"))
)

# --- combine ---
result = wallet_stats.join(order_stats)

# --- derived metric ---
result["fill_ratio"] = (
    result["triggers_with_fill"] / result["total_triggers"]
).fillna(0)

result.sort_values("total_pnl", ascending=False).head()

,total_triggers,total_fills,total_fill_pnl,total_trade_pnl,total_pnl,triggers_with_fill,fill_ratio
wallet,,,,,,,
0xcf6a714618a328c608a1c70cb62a31a6bef3f9d0,554,1100,20116.462863,55685.247070,20116.462863,422,0.761733
0x6bc74c392c320cfe10d5be61db978a58c8444ad4,303,388,14995.709984,36636.720180,14995.709984,178,0.587459
0x68c24bf4a8ad4d79a6fe4b8eec6f93a02dfd1711,377,534,878.542261,23429.830418,878.542261,263,0.697613
0x0cb10c40b0776e9ee8cef970af85724654dda76c,595,355,807.137790,8803.265493,807.137790,152,0.255462
0x1c1e841584db14084e10e7dca2ad0ab7b60dbfe7,57,143,722.055227,-631.627074,722.055227,50,0.877193


In [187]:
len(df)

29496

In [188]:
# get diff between trigger and fill timestamp per order_id
# without lambda, using groupby + agg + merge
trigger_df = df[df['row_type'] == 'trigger'].groupby('order_id')['dt'].min().reset_index()
fill_df = df[df['row_type'] == 'fill'].groupby('order_id')['dt'].min().reset_index()

merged_df = pd.merge(trigger_df, fill_df, on='order_id', how='outer', suffixes=('_trigger', '_fill'))
merged_df['diff_dt'] = merged_df['dt_fill'] - merged_df['dt_trigger']

In [189]:
merged_df[merged_df['diff_dt'].notnull()]['diff_dt'].mean().total_seconds()

53.805194

In [190]:
result[result['total_fill_pnl'] < 0].sort_values("total_fill_pnl").index.tolist()

['0x00ffb912f1bf01c0123869ddb92d4b6a6ad3f7ad',
 '0xf2c29fa94f5a18b1c68c00baee66155b89254de7',
 '0x8b72cb885a6bd4ea9d1da393ca231f0fa3476dbe',
 '0xdfe3fedc5c7679be42c3d393e99d4b55247b73c4',
 '0x7056bee200da86ee0a58f0e545250a29802088fc',
 '0x1c144e30f405a25f991cbd8baa15d40599090869',
 '0x97b20958ebed39b14f95dc6b454414ff082a1c44',
 '0x002dcd37b0b8fa8db98236e599fe1b90d6272561',
 '0x210e1d92cfacc18699a2527e705c8bc6828c5fd3',
 '0x865ab91c322793309506adca4f8f245e0023901b',
 '0x81a60a0882a2393dd4a453e257af763e3362762e',
 '0xc757850237e266d3b08d16b8455777a88590ccfc']

In [191]:
# Per-wallet PnL breakdown
if not fills.empty:
    wallet_pnl_df = (
        fills.groupby("wallet")
        .agg(
            filled_orders=("order_id", "nunique"),
            fill_rows=("order_id", "count"),
            net_pnl_usdc=("fill_pnl_usdc", "sum"),
            total_qty=("fill_qty", "sum"),
            win_rate=("token_winner", lambda s: fills.loc[s.index].groupby("order_id")["token_winner"].first().mean()),
        )
        .assign(win_rate=lambda d: (d["win_rate"] * 100).round(1))
        .sort_values("net_pnl_usdc", ascending=False)
        .reset_index()
    )
    display(wallet_pnl_df)


,wallet,filled_orders,fill_rows,net_pnl_usdc,total_qty,win_rate
0,0xcf6a714618a328c608a1c70cb62a31a6bef3f9d0,422,1100,20116.462863,108943.788408,56.9
1,0x6bc74c392c320cfe10d5be61db978a58c8444ad4,178,388,14995.709984,26916.747660,86.5
2,0x68c24bf4a8ad4d79a6fe4b8eec6f93a02dfd1711,263,534,878.542261,22833.057687,79.8
3,0x0cb10c40b0776e9ee8cef970af85724654dda76c,152,355,807.137790,11065.156518,47.4
4,0x1c1e841584db14084e10e7dca2ad0ab7b60dbfe7,50,143,722.055227,15202.692216,54.0
5,0x9e8ecc4cb3c4e48f544cba2fbbb252a6a65e8db8,450,1240,509.406839,23693.011186,59.6
6,0xd4549c366965829bde8efdae823ff767f250b47f,752,1760,406.744435,24176.583660,44.7
7,0x20a89c60befe1e2d866042deed93ab72be6fd960,871,1265,347.489585,7474.220198,60.0
8,0xdc32b242c97ad4c34151f690c4abb4ebe2f400bf,289,743,268.841218,8761.202014,52.6
9,0x520b4a0452005517964ee864985f2026d0dc100a,254,442,222.512116,10912.605843,45.3


In [192]:
wallet_pnl_df.head(10)['wallet'].tolist()

['0xcf6a714618a328c608a1c70cb62a31a6bef3f9d0',
 '0x6bc74c392c320cfe10d5be61db978a58c8444ad4',
 '0x68c24bf4a8ad4d79a6fe4b8eec6f93a02dfd1711',
 '0x0cb10c40b0776e9ee8cef970af85724654dda76c',
 '0x1c1e841584db14084e10e7dca2ad0ab7b60dbfe7',
 '0x9e8ecc4cb3c4e48f544cba2fbbb252a6a65e8db8',
 '0xd4549c366965829bde8efdae823ff767f250b47f',
 '0x20a89c60befe1e2d866042deed93ab72be6fd960',
 '0xdc32b242c97ad4c34151f690c4abb4ebe2f400bf',
 '0x520b4a0452005517964ee864985f2026d0dc100a']

In [193]:
# Orders that were NOT filled at all
filled_order_ids = set(fills["order_id"].unique()) if not fills.empty else set()
unfilled_triggers = triggers[~triggers["order_id"].isin(filled_order_ids)]
print(f"Unfilled orders    : {len(unfilled_triggers):,} ({len(unfilled_triggers)/max(len(triggers),1)*100:.1f}% of all triggers)")

# Partially filled orders (filled but qty_filled < qty_ordered)
if not fills.empty:
    filled_trig = triggers[triggers["order_id"].isin(filled_order_ids)].copy()
    partial_mask = filled_trig["qty_filled"] < filled_trig["qty_ordered"] - 1e-9
    partial_fills = filled_trig[partial_mask]
    print(f"Partially filled   : {len(partial_fills):,} ({len(partial_fills)/max(len(filled_trig),1)*100:.1f}% of filled orders)")
    print(f"Fully filled       : {len(filled_trig)-len(partial_fills):,}")

if not unfilled_triggers.empty:
    print("\nUnfilled breakdown by side:")
    display(unfilled_triggers.groupby("side").size().rename("count").reset_index())


Unfilled orders    : 6,494 (47.8% of all triggers)
Partially filled   : 1,363 (19.2% of filled orders)
Fully filled       : 5,721

Unfilled breakdown by side:


,side,count
0,BUY,6494


In [194]:
triggers[triggers['order_id'] == 34]

,row_type,order_id,wallet,condition_id,outcome,side,dt,tx_hash,price,order_price,qty_ordered,qty_filled,fill_qty,fill_source,token_winner,exec_price,fill_pnl_usdc,wallet_trade_pnl,shard
58,trigger,34,0x81a60a0882a2393dd4a453e257af763e3362762e,0x35a841022766818843e93a760e427f294c0219da347160d122cd94980a2df790,Yes,BUY,2026-02-16 05:06:41+00:00,0xd3e02d0cf5df3b0c0ce28f66cf070747ccbe5ebd0a75ae45b7909c38091b8ca2,0.001,0.001,524.48,63.763,NaN,None,False,NaN,NaN,-0.52448,3


In [195]:
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)
fills[fills['order_id'] == 34]

,row_type,order_id,wallet,condition_id,outcome,side,dt,tx_hash,price,order_price,qty_ordered,qty_filled,fill_qty,fill_source,token_winner,exec_price,fill_pnl_usdc,wallet_trade_pnl,shard
85,fill,34,0x81a60a0882a2393dd4a453e257af763e3362762e,0x35a841022766818843e93a760e427f294c0219da347160d122cd94980a2df790,Yes,BUY,2026-02-16 05:11:23+00:00,0x9c5fde581cb515e6c0d96ef41ea17fad11e0dd4277a48f1f16afbc491cc3cc2d,0.001,NaN,524.48,NaN,63.763,same_token,False,0.001,-0.063827,NaN,3


In [196]:
fills.groupby("wallet")["fill_pnl_usdc"].sum().sort_values(ascending=False).head(100)

wallet
0xcf6a714618a328c608a1c70cb62a31a6bef3f9d0    20116.462863
0x6bc74c392c320cfe10d5be61db978a58c8444ad4    14995.709984
0x68c24bf4a8ad4d79a6fe4b8eec6f93a02dfd1711      878.542261
0x0cb10c40b0776e9ee8cef970af85724654dda76c      807.137790
0x1c1e841584db14084e10e7dca2ad0ab7b60dbfe7      722.055227
0x9e8ecc4cb3c4e48f544cba2fbbb252a6a65e8db8      509.406839
0xd4549c366965829bde8efdae823ff767f250b47f      406.744435
0x20a89c60befe1e2d866042deed93ab72be6fd960      347.489585
0xdc32b242c97ad4c34151f690c4abb4ebe2f400bf      268.841218
0x520b4a0452005517964ee864985f2026d0dc100a      222.512116
0xd38b71f3e8ed1af71983e5c309eac3dfa9b35029      165.171955
0x7656ed7f597a0a61cd307591db198a42b2a7194b      121.774441
0x66c1a6fe836ff555ca32848646acedbbe93bfa3f       80.320070
0x5042e6dc8a612c493881a3e67519cc09f5f4fcb0       74.355615
0x672225e5e1aba1c970ac613efd1505f4b7a10762       34.721526
0x0b652d3e55be0e1d50c2f7be39358585c415293e       30.423270
0x99984e22205053950eb25453779267bcc1aee858       

In [197]:
trigger_wallets = triggers[["order_id", "wallet"]].set_index("order_id")["wallet"]

fills_with_wallet = fills.merge(trigger_wallets, on="order_id", how="left", suffixes=("", "_trigger"))

fills_with_wallet['notional'] = fills_with_wallet['fill_qty'] * fills_with_wallet['exec_price']

fills_with_wallet.groupby(["wallet", "condition_id"]).agg(
    pnl=("fill_pnl_usdc", "sum"),
    orders_filled=("order_id", "nunique")
    ).sort_values(['wallet', "pnl"], ascending=[True, False]).head(1)

,,pnl,orders_filled
wallet,condition_id,,
0x002dcd37b0b8fa8db98236e599fe1b90d6272561,0x9fdbd8e33ba9c5ff52b9caa343a277cd95e2d9661f1bd664b4e6b86b8b558490,47.012577,4


In [198]:
fills_with_wallet.head()

,row_type,order_id,wallet,condition_id,outcome,side,dt,tx_hash,price,order_price,...,qty_filled,fill_qty,fill_source,token_winner,exec_price,fill_pnl_usdc,wallet_trade_pnl,shard,wallet_trigger,notional
0,fill,6,0xc757850237e266d3b08d16b8455777a88590ccfc,0x905bd0615b38b035e60e69a46e0f185729ace5825a92f6dc39aede1d3ad31a13,No,BUY,2026-02-16 01:07:49+00:00,0x1e9ea7093aabffc14323905ab357516c1f2d00d882b28e1fdf0f571f6e098ae9,0.998,NaN,...,NaN,100.200401,same_token,True,0.998000,0.100401,NaN,9,0xc757850237e266d3b08d16b8455777a88590ccfc,100.000000
1,fill,7,0x66c1a6fe836ff555ca32848646acedbbe93bfa3f,0x8d7ff276b2a74beed4a15fd0f1744ecf05cfad7ed0c69aa01310e24b3b820bb3,Yes,BUY,2026-02-16 01:08:13+00:00,0xe65c3fda94eb29e72cf6d92156770ea4526855a090e059fa59b545870cbe52ec,0.350,NaN,...,NaN,14.000000,same_token,False,0.350712,-4.914874,NaN,8,0x66c1a6fe836ff555ca32848646acedbbe93bfa3f,4.909964
2,fill,7,0x66c1a6fe836ff555ca32848646acedbbe93bfa3f,0x8d7ff276b2a74beed4a15fd0f1744ecf05cfad7ed0c69aa01310e24b3b820bb3,Yes,BUY,2026-02-16 01:08:27+00:00,0x85ea04df68e06790060bba70ea471b8aae14b1f7c8fbad760dc53528b1a39925,0.350,NaN,...,NaN,0.250000,same_token,False,0.350712,-0.087766,NaN,8,0x66c1a6fe836ff555ca32848646acedbbe93bfa3f,0.087678
3,fill,7,0x66c1a6fe836ff555ca32848646acedbbe93bfa3f,0x8d7ff276b2a74beed4a15fd0f1744ecf05cfad7ed0c69aa01310e24b3b820bb3,Yes,BUY,2026-02-16 01:08:31+00:00,0xaad229554539116ef7b00ff29d4ad891dbd108dc07d64a7ec2a3bef067e39b40,0.340,NaN,...,NaN,10.640000,opposite_token,False,0.350712,-3.735304,NaN,8,0x66c1a6fe836ff555ca32848646acedbbe93bfa3f,3.731573
4,fill,7,0x66c1a6fe836ff555ca32848646acedbbe93bfa3f,0x8d7ff276b2a74beed4a15fd0f1744ecf05cfad7ed0c69aa01310e24b3b820bb3,Yes,BUY,2026-02-16 01:08:31+00:00,0xaad229554539116ef7b00ff29d4ad891dbd108dc07d64a7ec2a3bef067e39b40,0.340,NaN,...,NaN,5.000000,opposite_token,False,0.350712,-1.755312,NaN,8,0x66c1a6fe836ff555ca32848646acedbbe93bfa3f,1.753559


In [199]:
wallet_pnls = (
    fills_with_wallet
    .groupby(["wallet", "condition_id"])
    .agg(
        pnl=("fill_pnl_usdc", "sum"),
        orders_filled=("order_id", "nunique"),
        notional=("notional", "sum"),
    )
    .groupby("wallet")
    .agg(
        pnl=("pnl", "sum"),
        notional=("notional", "sum"),
        unique_conditions=("pnl", "size"),  # ✅ count rows
        total_orders_filled=("orders_filled", "sum"),
    )
    .sort_values("pnl", ascending=False)
)

In [200]:
wallet_pnls.head()

,pnl,notional,unique_conditions,total_orders_filled
wallet,,,,
0xcf6a714618a328c608a1c70cb62a31a6bef3f9d0,20116.462863,15809.985094,22,422
0x6bc74c392c320cfe10d5be61db978a58c8444ad4,14995.709984,2589.849190,30,178
0x68c24bf4a8ad4d79a6fe4b8eec6f93a02dfd1711,878.542261,12318.722206,55,263
0x0cb10c40b0776e9ee8cef970af85724654dda76c,807.137790,2796.413671,72,152
0x1c1e841584db14084e10e7dca2ad0ab7b60dbfe7,722.055227,1854.608710,8,50


In [201]:
MAX_WALLET_EXPOSURE = 1000.0  # USDC
wallet_pnls['pnl_limited'] = np.where(
    wallet_pnls['notional'] <= MAX_WALLET_EXPOSURE,
    wallet_pnls['pnl'],
    wallet_pnls['pnl'] * MAX_WALLET_EXPOSURE / wallet_pnls['notional']
)

In [202]:
wallet_pnls

,pnl,notional,unique_conditions,total_orders_filled,pnl_limited
wallet,,,,,
0xcf6a714618a328c608a1c70cb62a31a6bef3f9d0,20116.462863,15809.985094,22,422,1272.389743
0x6bc74c392c320cfe10d5be61db978a58c8444ad4,14995.709984,2589.849190,30,178,5790.186565
0x68c24bf4a8ad4d79a6fe4b8eec6f93a02dfd1711,878.542261,12318.722206,55,263,71.317645
0x0cb10c40b0776e9ee8cef970af85724654dda76c,807.137790,2796.413671,72,152,288.633187
0x1c1e841584db14084e10e7dca2ad0ab7b60dbfe7,722.055227,1854.608710,8,50,389.330226
0x9e8ecc4cb3c4e48f544cba2fbbb252a6a65e8db8,509.406839,11822.861844,255,450,43.086593
0xd4549c366965829bde8efdae823ff767f250b47f,406.744435,10461.120636,559,752,38.881536
0x20a89c60befe1e2d866042deed93ab72be6fd960,347.489585,3888.899393,266,871,89.354223
0xdc32b242c97ad4c34151f690c4abb4ebe2f400bf,268.841218,4406.800590,205,289,61.005987


In [203]:
result = (
    fills_with_wallet
    # 1️⃣ PnL per market (condition) per wallet
    .groupby(["wallet", "condition_id"])["fill_pnl_usdc"]
    .sum()
    
    # 2️⃣ Then per wallet
    .groupby("wallet")
    .agg(
        unique_conditions="count",   # number of markets traded
        median_market_pnl="median",  # median PnL across markets
    )
    .sort_values("unique_conditions", ascending=False)
    .head(20)
)

In [204]:
result.sort_values("median_market_pnl", ascending=False).head(100)

,unique_conditions,median_market_pnl
wallet,,
0xdc32b242c97ad4c34151f690c4abb4ebe2f400bf,205,6.034444
0x865ab91c322793309506adca4f8f245e0023901b,89,4.702943
0x6bc74c392c320cfe10d5be61db978a58c8444ad4,30,3.363448
0xc757850237e266d3b08d16b8455777a88590ccfc,79,2.674923
0x68c24bf4a8ad4d79a6fe4b8eec6f93a02dfd1711,55,0.919598
0x66c1a6fe836ff555ca32848646acedbbe93bfa3f,133,0.851160
0x20a89c60befe1e2d866042deed93ab72be6fd960,266,0.804810
0x0b652d3e55be0e1d50c2f7be39358585c415293e,65,0.800900
0x9e8ecc4cb3c4e48f544cba2fbbb252a6a65e8db8,255,0.491970


In [205]:
fills_with_wallet.head()

,row_type,order_id,wallet,condition_id,outcome,side,dt,tx_hash,price,order_price,...,qty_filled,fill_qty,fill_source,token_winner,exec_price,fill_pnl_usdc,wallet_trade_pnl,shard,wallet_trigger,notional
0,fill,6,0xc757850237e266d3b08d16b8455777a88590ccfc,0x905bd0615b38b035e60e69a46e0f185729ace5825a92f6dc39aede1d3ad31a13,No,BUY,2026-02-16 01:07:49+00:00,0x1e9ea7093aabffc14323905ab357516c1f2d00d882b28e1fdf0f571f6e098ae9,0.998,NaN,...,NaN,100.200401,same_token,True,0.998000,0.100401,NaN,9,0xc757850237e266d3b08d16b8455777a88590ccfc,100.000000
1,fill,7,0x66c1a6fe836ff555ca32848646acedbbe93bfa3f,0x8d7ff276b2a74beed4a15fd0f1744ecf05cfad7ed0c69aa01310e24b3b820bb3,Yes,BUY,2026-02-16 01:08:13+00:00,0xe65c3fda94eb29e72cf6d92156770ea4526855a090e059fa59b545870cbe52ec,0.350,NaN,...,NaN,14.000000,same_token,False,0.350712,-4.914874,NaN,8,0x66c1a6fe836ff555ca32848646acedbbe93bfa3f,4.909964
2,fill,7,0x66c1a6fe836ff555ca32848646acedbbe93bfa3f,0x8d7ff276b2a74beed4a15fd0f1744ecf05cfad7ed0c69aa01310e24b3b820bb3,Yes,BUY,2026-02-16 01:08:27+00:00,0x85ea04df68e06790060bba70ea471b8aae14b1f7c8fbad760dc53528b1a39925,0.350,NaN,...,NaN,0.250000,same_token,False,0.350712,-0.087766,NaN,8,0x66c1a6fe836ff555ca32848646acedbbe93bfa3f,0.087678
3,fill,7,0x66c1a6fe836ff555ca32848646acedbbe93bfa3f,0x8d7ff276b2a74beed4a15fd0f1744ecf05cfad7ed0c69aa01310e24b3b820bb3,Yes,BUY,2026-02-16 01:08:31+00:00,0xaad229554539116ef7b00ff29d4ad891dbd108dc07d64a7ec2a3bef067e39b40,0.340,NaN,...,NaN,10.640000,opposite_token,False,0.350712,-3.735304,NaN,8,0x66c1a6fe836ff555ca32848646acedbbe93bfa3f,3.731573
4,fill,7,0x66c1a6fe836ff555ca32848646acedbbe93bfa3f,0x8d7ff276b2a74beed4a15fd0f1744ecf05cfad7ed0c69aa01310e24b3b820bb3,Yes,BUY,2026-02-16 01:08:31+00:00,0xaad229554539116ef7b00ff29d4ad891dbd108dc07d64a7ec2a3bef067e39b40,0.340,NaN,...,NaN,5.000000,opposite_token,False,0.350712,-1.755312,NaN,8,0x66c1a6fe836ff555ca32848646acedbbe93bfa3f,1.753559
